In [ ]:
# Google Drive mount and project-root setup for Colab
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/phase_conditioned_diffusion_policy')

try:
    from google.colab import drive
except ImportError:
    print(f'Not running in Google Colab; keeping current working directory: {Path.cwd()}')
else:
    drive.mount('/content/drive')
    if not PROJECT_DIR.exists():
        raise FileNotFoundError(
            f'Expected project directory not found: {PROJECT_DIR}\n'
            'Update PROJECT_DIR to the Google Drive folder that contains this repository.'
        )
    os.chdir(PROJECT_DIR)
    print(f'Current working directory: {Path.cwd()}')


In [ ]:
# Colab dependency setup
import sys
import subprocess

try:
    import google.colab  # noqa: F401
except ImportError:
    print("Not running in Google Colab; skipping dependency installation and using the current environment.")
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "."])

    import gymnasium as gym
    import mujoco
    import minari
    import torch

    gym.make("Ant-v5").close()

    print(f"gymnasium={gym.__version__}")
    print(f"mujoco={mujoco.__version__}")
    print(f"minari={minari.__version__}")
    print(f"torch={torch.__version__}")
    print("Ant-v5 environment smoke check passed.")


# 01 — Data Preparation

Minari/D4RL Ant dataset에서 trajectory를 다운로드하고, Hilbert transform 기반 phase label과 quality filter를 적용해 `demos_ant.npz`를 생성한 뒤, train/val split과 normalization stats를 `norm_stats.npz`로 저장합니다.

이 노트북은 **실행 orchestration만 담당**합니다. 핵심 로직은 `pcdp/data_extraction.py`, `pcdp/data_pipeline.py`, artifact 경로 관리는 `pcdp/paths.py`에 있습니다.

Colab에서는 실행 전에 Drive mount(필요 시), repository root로 `%cd`, dependency 설치를 완료했다고 가정합니다. MuJoCo 렌더링 시스템 패키지는 README의 설치 섹션을 참고하세요.

## 1. Artifact paths


In [ ]:
from pcdp.paths import ARTIFACT_ROOT, DATA_DIR, FIGURES_DIR, ensure_artifact_dirs
ensure_artifact_dirs()
print(f'✓ artifact root: {ARTIFACT_ROOT}')


## 2. Dataset discovery and load

In [ ]:
import minari
import numpy as np

from pcdp.data_extraction import (
    ANT_PHASE_JOINT_IDX,
    DEFAULT_MINARI_ANT_DATASET,
    DemoExtractionConfig,
    extract_demos_from_episodes,
    load_ant_dataset,
    materialize_episodes,
    plot_demo_quality,
    print_demo_quality_report,
    save_demos,
)

print(f'Minari version: {minari.__version__}')
dataset = load_ant_dataset(minari, DEFAULT_MINARI_ANT_DATASET)


## 3. Episode diagnostics

In [ ]:
config = DemoExtractionConfig()
episodes = materialize_episodes(dataset, expected_obs_dim=config.expected_obs_dim)
print(f'Phase 라벨링용 joint: obs[{ANT_PHASE_JOINT_IDX}]')


## 4. Demo extraction and `demos_ant.npz` save

In [ ]:
demos = extract_demos_from_episodes(episodes, config)
assert demos is not None, '선택된 demos가 없습니다. DemoExtractionConfig 임계값을 완화해 주세요.'
demos_path = save_demos(demos, DATA_DIR / 'demos_ant.npz')


## 5. Quality plots and report

In [ ]:
demos_loaded = np.load(demos_path)
quality_path, visualization_path = plot_demo_quality(demos_loaded, FIGURES_DIR)
print_demo_quality_report(demos_loaded)


## 6. Data pipeline — split + normalization

Phase 1에서 만든 `demos_ant.npz`를 episode-level train/val split하고, train-only normalization stats와 horizon/frequency 메타데이터를 `norm_stats.npz`로 저장합니다.

In [ ]:
from pcdp.data_pipeline import DataPipelineConfig, plot_phase_advance, run_data_pipeline

DATA_PATH = DATA_DIR / 'demos_ant.npz'
NORM_PATH = DATA_DIR / 'norm_stats.npz'
assert DATA_PATH.exists(), f'파일 없음: {DATA_PATH}'

pipeline_cfg = DataPipelineConfig(
    obs_horizon=2,
    pred_horizon=16,
    action_horizon=8,
    val_ratio=0.15,
    batch_size=256,
    num_workers=2,
    seed=42,
)
pipeline = run_data_pipeline(DATA_PATH, NORM_PATH, pipeline_cfg)
train_dataset = pipeline['train_dataset']
val_dataset = pipeline['val_dataset']
train_loader = pipeline['train_loader']
val_loader = pipeline['val_loader']


## 6. Phase-advance diagnostic

In [ ]:
phase_advance = plot_phase_advance(
    train_dataset,
    FIGURES_DIR,
    f_mean=pipeline['project_data']['freq_window_mean'],
    seed=pipeline_cfg.seed,
)
print('\n=== 01 Data Preparation 완료 ===')
print(f'Demos:        {demos_path}')
print(f'Norm stats:   {NORM_PATH}')
print(f'Train chunks: {len(train_dataset)}')
print(f'Val chunks:   {len(val_dataset)}')
print(f'Steps/epoch:  {len(train_loader)}')
print(f'Phase plot:   {phase_advance["path"]}')
